# LISTA-CP-KAN — Kaggle Notebook

**Important — preventing weight loss on runtime reset:**
- All checkpoints save to `/kaggle/working/` automatically.
- Checkpoints survive a runtime reset — just re-run from Cell 1.
- Training resumes automatically from the last completed layer/phase.
- After training, use **Save Version → Save & Run All** to commit outputs permanently.

**Steps:**
1. Install dependencies
2. Upload / copy project files
3. Train baseline (16-layer greedy, ~15-25 min on P100/T4)
4. Train KAN variants (~10-15 min each)
5. Evaluate and plot

In [ ]:
# Cell 1: Install dependencies
!pip install pykan==0.2.8 -q
import torch, numpy as np
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 2: Set up working directory and show existing checkpoints
import os

WORK_DIR = '/kaggle/working'
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

# Show any checkpoints already saved (from a previous run)
ckpts = [f for f in os.listdir(WORK_DIR) if f.endswith('.pth')]
if ckpts:
    print(f'Existing checkpoints ({len(ckpts)}):')
    for c in sorted(ckpts):
        size_mb = os.path.getsize(os.path.join(WORK_DIR, c)) / 1e6
        print(f'  {c}  ({size_mb:.1f} MB)')
else:
    print('No checkpoints found. Training will start from scratch.')

In [ ]:
# Cell 3: Copy project .py files into /kaggle/working
# Option A: files are in a Kaggle Dataset attached to this notebook
#   (Upload them via Add Data → New Dataset before running)
import shutil, glob

# Look for .py files in attached datasets
src_dirs = glob.glob('/kaggle/input/*/') + glob.glob('/kaggle/input/*/*/')
py_files_found = []
for d in src_dirs:
    for f in glob.glob(os.path.join(d, '*.py')):
        dst = os.path.join(WORK_DIR, os.path.basename(f))
        shutil.copy2(f, dst)
        py_files_found.append(os.path.basename(f))
    # Also copy LISTA-CPSS subfolder if present
    lista_src = os.path.join(d, 'LISTA-CPSS')
    if os.path.isdir(lista_src):
        shutil.copytree(lista_src, os.path.join(WORK_DIR, 'LISTA-CPSS'),
                        dirs_exist_ok=True)
        print('Copied LISTA-CPSS/ folder.')

if py_files_found:
    print(f'Copied .py files: {py_files_found}')
else:
    # Option B: paste files directly — uncomment and adapt as needed
    print('No .py files found in attached datasets.')
    print('Please attach your dataset containing the .py files (see notebook instructions).')

In [ ]:
# Cell 4: Verify imports
import sys
sys.path.insert(0, WORK_DIR)

from problem_setup import random_A, gen_samples_torch, compute_nmse_db_torch
from lista_cp_torch import LISTACPTorch
from lista_cp_kan import LISTACPKan
from kan_theta_predictor import KANThetaPredictor
print('All imports successful.')

## Step 1 — Train Baseline

- Saves a checkpoint after **each completed layer** to `/kaggle/working/lista_cp_layer01.pth` … `lista_cp_layer16.pth`.
- If the runtime resets, **just re-run this cell** — it will detect the latest layer checkpoint and resume automatically.
- Takes ~15-25 min on a T4/P100 GPU.

In [ ]:
# Cell 5: Train baseline (resumes automatically from last checkpoint)
!python /kaggle/working/train_baseline.py

In [ ]:
# Cell 6: Verify baseline checkpoint
ckpt = torch.load('/kaggle/working/lista_cp_baseline.pth',
                  map_location='cpu', weights_only=False)
print(f"Baseline thetas: {[round(v, 5) for v in ckpt['thetas']]}")
print(f"Test NMSE per layer: {[round(v, 2) for v in ckpt['test_nmse_per_layer']]}")
print(f"Final layer 16 NMSE: {ckpt['test_nmse_per_layer'][-1]:.2f} dB")

## Step 2 — Train KAN Variants

- Each variant saves a final checkpoint to `/kaggle/working/lista_cp_kan1_final.pth` or `kan3_final.pth`.
- If interrupted, re-running resumes from Phase B (Phase A warmup is skipped if its best checkpoint exists).
- Takes ~10-15 min each.

In [ ]:
# Cell 7: Train 1norm variant
!python /kaggle/working/train_kan.py --variant 1norm

In [ ]:
# Cell 8: Train 3norm variant
!python /kaggle/working/train_kan.py --variant 3norm

## Step 3 — Evaluate

In [ ]:
# Cell 9: Run full evaluation
!python /kaggle/working/evaluate.py

In [ ]:
# Cell 10: Display result plots
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display
import glob

plot_files = sorted(glob.glob('/kaggle/working/results/*.png'))
for pf in plot_files:
    print(f'\n--- {os.path.basename(pf)} ---')
    img = mpimg.imread(pf)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img); ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 11: List all saved files (checkpoints + plots)
print('Checkpoints:')
for f in sorted(glob.glob('/kaggle/working/*.pth')):
    print(f'  {os.path.basename(f):50s} {os.path.getsize(f)/1e6:.1f} MB')
print('\nPlots:')
for f in sorted(glob.glob('/kaggle/working/results/*.png')):
    print(f'  {os.path.basename(f)}')